CELL 1 — Imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

CELL 2 — Project Configuration

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/expo"

ARTIFACT_DIR = f"{PROJECT_ROOT}/artifacts"

os.makedirs(
    ARTIFACT_DIR,
    exist_ok=True
)

CELL 3 — Load Artifacts

In [ ]:
reviews = pd.read_parquet(
    f"{ARTIFACT_DIR}/reviews_clean.parquet"
)

aspect_df = pd.read_parquet(
    f"{ARTIFACT_DIR}/review_aspects.parquet"
)

profiles = pd.read_parquet(
    f"{ARTIFACT_DIR}/profiles_clean.parquet"
)

hotel_stats = pd.read_parquet(
    f"{ARTIFACT_DIR}/hotel_stats.parquet"
)

monthly_hotel_stats = pd.read_parquet(
    f"{ARTIFACT_DIR}/monthly_hotel_stats.parquet"
)

print(reviews.shape)
print(aspect_df.shape)

(50000, 13)
(88338, 6)


CELL 4 — Merge Aspect Metadata

In [ ]:
aspect_df = aspect_df.merge(
    reviews[
        [
            "review_id",
            "hotel_id",
            "rating",
            "review_date",
            "traveler_type",
            "verified"
        ]
    ],
    on=[
        "review_id",
        "hotel_id"
    ],
    how="left"
)

aspect_df.head()

,aspect,confidence,sentiment,sentence,review_id,hotel_id,rating,review_date,traveler_type,verified
0,service,0.6911,0.0000,Pure luxury from check-in to checkout — utterl...,R31838,H051,4,2024-11-23,None,False
1,accessibility,0.7030,0.0000,"Fully step-free with a spacious, well-designed...",R31838,H051,4,2024-11-23,None,False
2,cleanliness,0.8005,-0.3089,Room wasn't clean on arrival — found hair in t...,R31838,H051,4,2024-11-23,None,False
3,family,0.7460,0.8225,The kids loved the pool and the staff were won...,R25484,H037,4,2025-05-19,None,True
4,service,0.6490,0.8225,The kids loved the pool and the staff were won...,R25484,H037,4,2025-05-19,None,True


CELL 5 — Normalize Sentiment

In [ ]:
aspect_df["aspect_score"] = (
    (
        aspect_df["sentiment"]
        + 1
    ) / 2
) * 5

CELL 6 — Confidence Weighting

In [ ]:
aspect_df["weighted_score"] = (
    aspect_df["aspect_score"]
    *
    aspect_df["confidence"]
)

CELL 7 — Core Hotel Feature Store

In [ ]:
hotel_feature_store = (
    aspect_df
    .groupby(
        [
            "hotel_id",
            "aspect"
        ]
    )
    .agg(
        mean_sentiment=(
            "aspect_score",
            "mean"
        ),

        mention_count=(
            "aspect",
            "count"
        ),

        avg_confidence=(
            "confidence",
            "mean"
        ),

        score_std=(
            "aspect_score",
            "std"
        )
    )
    .reset_index()
)

# -----------------------------------------------------
# Confidence-adjusted score
# Confidence should influence, not dominate.
# -----------------------------------------------------

hotel_feature_store["mean_score"] = (
    hotel_feature_store["mean_sentiment"]
    *
    (
        0.70
        +
        0.30
        * hotel_feature_store["avg_confidence"]
    )
)

# -----------------------------------------------------
# Consistency score
# Lower variance means more reliable hotel experience.
# -----------------------------------------------------

hotel_feature_store["consistency_score"] = (
    1
    /
    (
        1
        +
        hotel_feature_store["score_std"]
        .fillna(0)
    )
)

display(
    hotel_feature_store.head()
)

,hotel_id,aspect,mean_sentiment,mention_count,avg_confidence,score_std,mean_score,consistency_score
0,H001,accessibility,3.058293,99,0.757784,0.910046,2.836063,0.523548
1,H001,beach,2.163172,145,0.790794,0.369787,2.027408,0.730040
2,H001,cleanliness,3.372752,113,0.773486,0.912804,3.143559,0.522793
3,H001,culture,3.386044,130,0.797000,0.895777,3.179834,0.527488
4,H001,family,3.018780,83,0.776178,1.326811,2.816080,0.429773


CELL 8 — Create Matrix Representation

In [ ]:
hotel_matrix = (
    hotel_feature_store
    .pivot(
        index="hotel_id",
        columns="aspect",
        values="mean_score"
    )
)

# -----------------------------------------------------
# Aspect-specific priors instead of hardcoded 2.5
# -----------------------------------------------------

for col in hotel_matrix.columns:

    aspect_prior = (
        hotel_matrix[col]
        .mean()
    )

    hotel_matrix[col] = (
        hotel_matrix[col]
        .fillna(
            aspect_prior
        )
    )

# -----------------------------------------------------
# Additional matrices for later notebooks
# -----------------------------------------------------

consistency_matrix = (
    hotel_feature_store
    .pivot(
        index="hotel_id",
        columns="aspect",
        values="consistency_score"
    )
)

for col in consistency_matrix.columns:

    consistency_matrix[col] = (
        consistency_matrix[col]
        .fillna(
            consistency_matrix[col]
            .mean()
        )
    )

mention_matrix = (
    hotel_feature_store
    .pivot(
        index="hotel_id",
        columns="aspect",
        values="mention_count"
    )
)

for col in mention_matrix.columns:

    mention_matrix[col] = (
        mention_matrix[col]
        .fillna(0)
    )

print(
    "Hotel Matrix Shape:",
    hotel_matrix.shape
)

display(
    hotel_matrix.head()
)

Hotel Matrix Shape: (120, 12)


aspect,accessibility,beach,business,cleanliness,culture,family,location,nightlife,safety,service,value,wellness
hotel_id,,,,,,,,,,,,
H001,2.836063,2.027408,2.880431,3.143559,3.179834,2.81608,1.905463,3.575800,2.746635,3.176890,3.018190,2.262341
H002,2.905924,3.639688,2.880431,3.041262,3.323240,2.19335,2.348207,3.575800,2.746635,2.672887,4.105985,4.160039
H003,3.202606,1.918169,2.695983,3.041262,3.176366,2.19335,3.549310,4.302797,2.746635,2.145487,3.016300,4.009839
H004,2.905924,3.418240,3.112819,3.245252,1.057728,2.19335,2.736542,3.575800,1.106327,2.542965,3.016300,3.466699
H005,2.905924,3.135460,2.880431,3.372808,3.158275,2.19335,2.374146,2.268175,3.390673,3.444286,2.242000,3.466699


CELL 9 — Confidence Scores

In [ ]:
hotel_confidence = (
    hotel_feature_store
    .groupby("hotel_id")
    ["mention_count"]
    .sum()
    .reset_index()
)

hotel_confidence[
    "confidence_score"
] = np.log1p(
    hotel_confidence[
        "mention_count"
    ]
)

hotel_confidence.head()

,hotel_id,mention_count,confidence_score
0,H001,800,6.685861
1,H002,355,5.874931
2,H003,340,5.831882
3,H004,1818,7.506042
4,H005,167,5.123964


CELL 10 — Aspect Trends

In [ ]:
aspect_df["review_date"] = pd.to_datetime(
    aspect_df["review_date"]
)

aspect_df["year_month"] = (
    aspect_df["review_date"]
    .dt.to_period("M")
    .astype(str)
)

In [ ]:
hotel_trends = (
    aspect_df
    .groupby(
        [
            "hotel_id",
            "aspect",
            "year_month"
        ]
    )
    .agg(
        monthly_score=(
            "weighted_score",
            "mean"
        )
    )
    .reset_index()
)

CELL 11 — Compute Trend Signals

In [ ]:
trend_results = []

for (
    hotel_id,
    aspect
), group in hotel_trends.groupby(
    [
        "hotel_id",
        "aspect"
    ]
):

    group = group.sort_values(
        "year_month"
    )

    if len(group) < 6:
        continue

    recent = (
        group.tail(6)
        ["monthly_score"]
        .mean()
    )

    older = (
        group.head(6)
        ["monthly_score"]
        .mean()
    )

    trend = recent - older

    trend_results.append({
        "hotel_id": hotel_id,
        "aspect": aspect,
        "trend_score": trend
    })

hotel_trend_scores = pd.DataFrame(
    trend_results
)

CELL 12 — Traveler Personalization Store

In [ ]:
traveler_feature_store = (
    aspect_df[
        aspect_df[
            "traveler_type"
        ]
        .notna()
    ]
    .groupby(
        [
            "hotel_id",
            "traveler_type",
            "aspect"
        ]
    )
    .agg(
        traveler_score=(
            "weighted_score",
            "mean"
        )
    )
    .reset_index()
)

CELL 13 — Contradiction Detection

In [ ]:
contradictions = []

for (
    hotel_id,
    aspect
), group in traveler_feature_store.groupby(
    [
        "hotel_id",
        "aspect"
    ]
):

    if len(group) < 2:
        continue

    diff = (
        group[
            "traveler_score"
        ].max()
        -
        group[
            "traveler_score"
        ].min()
    )

    if diff > 1:

        contradictions.append({

            "hotel_id":
                hotel_id,

            "aspect":
                aspect,

            "best_for":
                group.loc[
                    group[
                        "traveler_score"
                    ].idxmax(),
                    "traveler_type"
                ],

            "worst_for":
                group.loc[
                    group[
                        "traveler_score"
                    ].idxmin(),
                    "traveler_type"
                ],

            "difference":
                diff
        })

hotel_contradictions = pd.DataFrame(
    contradictions
)

CELL 14 — Diagnostics

In [ ]:
print(
    hotel_matrix.shape
)

print(
    hotel_confidence.describe()
)

print(
    hotel_contradictions.head()
)

(120, 12)
       mention_count  confidence_score
count     120.000000        120.000000
mean      736.150000          6.341634
std       512.696556          0.751902
min       139.000000          4.941642
25%       306.000000          5.726832
50%       581.000000          6.366434
75%      1076.250000          6.982057
max      2055.000000          7.628518
  hotel_id    aspect best_for worst_for  difference
0     H001    family  leisure    family    1.146835
1     H001   service  leisure    family    1.402817
2     H003  business   family    couple    1.431106
3     H003  wellness   couple    family    1.050932
4     H005   culture    group  business    1.033685


In [ ]:
# ==========================================================
# Consistency Matrix
# ==========================================================

consistency_matrix = (
    hotel_feature_store
    .pivot(
        index="hotel_id",
        columns="aspect",
        values="consistency_score"
    )
)

for col in consistency_matrix.columns:
    consistency_matrix[col] = (
        consistency_matrix[col]
        .fillna(
            consistency_matrix[col].mean()
        )
    )

# ==========================================================
# Mention Matrix
# ==========================================================

mention_matrix = (
    hotel_feature_store
    .pivot(
        index="hotel_id",
        columns="aspect",
        values="mention_count"
    )
)

mention_matrix = mention_matrix.fillna(0)

print("Consistency Matrix Shape:")
print(consistency_matrix.shape)

print()

print("Mention Matrix Shape:")
print(mention_matrix.shape)

Consistency Matrix Shape:
(120, 12)

Mention Matrix Shape:
(120, 12)


CELL 15 — Save Artifacts

In [ ]:
hotel_feature_store.to_parquet(
    f"{ARTIFACT_DIR}/hotel_feature_store.parquet",
    index=False
)

hotel_matrix.to_parquet(
    f"{ARTIFACT_DIR}/hotel_matrix.parquet"
)

hotel_trend_scores.to_parquet(
    f"{ARTIFACT_DIR}/hotel_trends.parquet",
    index=False
)

traveler_feature_store.to_parquet(
    f"{ARTIFACT_DIR}/traveler_feature_store.parquet",
    index=False
)

hotel_contradictions.to_parquet(
    f"{ARTIFACT_DIR}/hotel_contradictions.parquet",
    index=False
)

hotel_confidence.to_parquet(
    f"{ARTIFACT_DIR}/hotel_confidence_scores.parquet",
    index=False
)

consistency_matrix.to_parquet(
    f"{ARTIFACT_DIR}/consistency_matrix.parquet"
)

mention_matrix.to_parquet(
    f"{ARTIFACT_DIR}/mention_matrix.parquet"
)

print(
    "\nSaved Successfully\n"
)

print(
    os.listdir(
        ARTIFACT_DIR
    )
)


Saved Successfully

['reviews_clean.parquet', 'profiles_clean.parquet', 'hotel_categories.json', 'monthly_hotel_stats.parquet', 'hotel_stats.parquet', 'dataset_metadata.json', 'top_words.json', 'top_bigrams.json', 'hotel_review_counts.json', 'aspect_keywords.json', 'aspect_embeddings.npy', 'aspect_descriptions.json', 'review_aspects.parquet', 'hotel_feature_store.parquet', 'hotel_matrix.parquet', 'hotel_trends.parquet', 'traveler_feature_store.parquet', 'hotel_contradictions.parquet', 'hotel_confidence_scores.parquet', 'consistency_matrix.parquet', 'mention_matrix.parquet']


In [ ]:
hotel_confidence.describe()

,mention_count,confidence_score
count,120.000000,120.000000
mean,736.150000,6.341634
std,512.696556,0.751902
min,139.000000,4.941642
25%,306.000000,5.726832
50%,581.000000,6.366434
75%,1076.250000,6.982057
max,2055.000000,7.628518


In [ ]:
hotel_matrix.describe().T

,count,mean,std,min,25%,50%,75%,max
aspect,,,,,,,,
accessibility,120.0,2.905924,0.279154,2.003400,2.905924,2.905924,3.020389,3.413750
beach,120.0,3.135460,0.361643,1.918169,3.135460,3.135460,3.168403,3.656692
business,120.0,2.880431,0.684517,1.200625,2.880431,2.880431,3.374454,3.874536
cleanliness,120.0,3.041262,0.400647,1.914460,3.041262,3.041262,3.262263,3.643070
culture,120.0,3.176366,0.662515,1.057728,3.169724,3.176366,3.423393,4.227517
family,120.0,2.193350,0.774696,0.473142,2.193350,2.193350,2.193350,3.492499
location,120.0,2.736542,0.550698,1.687834,2.435326,2.736542,3.186624,3.689293
nightlife,120.0,3.575800,0.560236,2.268175,3.575800,3.575800,3.933996,4.345397
safety,120.0,2.746635,0.569897,1.106327,2.746635,2.746635,2.746635,3.603704
